In [ ]:
import os
from pathlib import Path
import pandas as pd
import numpy as np
import requests
from ira.ingest.ingest_scorecard import save
from ira.clean.clean_scorecard import clean
from ira.config import SCORECARD_KEY, BASE_URL
import math
import json
import time

root = Path("__file__").resolve().parents[1]
# os.chdir(root/"Seb_branch"/"institutional-roi-analysis"/"notebooks")
pd.set_option("display.max_columns",None)
display(root)

WindowsPath('C:/Users/sebas/PycharmProjects/Git')

In [ ]:
def collect(state: str = "FL", per_page: int = 100) -> pd.DataFrame:
    fields = ",".join([
        "id",
        "school.name",

        "location.lat",
        "location.lon",

        "latest.programs.cip_4_digit.code",
        "latest.programs.cip_4_digit.unit_id",
        "latest.programs.cip_4_digit.title",
        "latest.programs.cip_4_digit.school.type",
        "latest.programs.cip_4_digit.credential.level",
        "latest.programs.cip_4_digit.distance",

        "latest.school.locale",
        "latest.school.carnegie_size_setting",
        "latest.admissions.admission_rate.overall",
        "latest.student.demographics.median_family_income",
        "latest.student.students_with_pell_grant",
        "latest.school.open_admissions_policy",
        "latest.student.demographics.age_entry",
        "latest.school.title_iv.eligibility_type",
        "latest.programs.cip_4_digit.earnings.1_yr.overall_median_earnings",
        "latest.programs.cip_4_digit.earnings.1_yr.working_not_enrolled.overall_count",
        "latest.programs.cip_4_digit.earnings.2_yr.overall_median_earnings",
        "latest.programs.cip_4_digit.earnings.2_yr.working_not_enrolled.overall_count",
        "latest.programs.cip_4_digit.earnings.3_yr.overall_median_earnings",
        "latest.programs.cip_4_digit.earnings.3_yr.working_not_enrolled.overall_count",
        "latest.programs.cip_4_digit.earnings.4_yr.overall_median_earnings",
        "latest.programs.cip_4_digit.earnings.4_yr.working_not_enrolled.overall_count",
        "latest.programs.cip_4_digit.earnings.5_yr.overall_median_earnings",
        "latest.programs.cip_4_digit.earnings.5_yr.working_not_enrolled.overall_count"
    ])

    params = {
        "api_key": SCORECARD_KEY,
        "school.state": state,
        "fields": fields,
        "per_page": str(per_page),

        # only programs with earnings reported
        "latest.programs.cip_4_digit.earnings.4_yr.overall_median_earnings__range": "1.."
    }

    dfs = []

    params["page"] = "0"
    response = get_with_retries(BASE_URL, params=params, timeout=30)
    data = get_json_or_raise(response)

    total = int(data["metadata"]["total"])
    per_page_actual = int(data["metadata"]["per_page"])
    total_pages = math.ceil(total / per_page_actual)

    META = [
        "id",
        "school.name",
        "location.lat",
        "location.lon",
        "latest.school.locale",
        "latest.school.carnegie_size_setting",
        "latest.admissions.admission_rate.overall",
        "latest.student.demographics.median_family_income",
        "latest.student.students_with_pell_grant",
        "latest.school.open_admissions_policy",
        "latest.student.demographics.age_entry",
        "latest.school.title_iv.eligibility_type",
    ]

    def page_to_df(data):
        results = [r for r in data.get("results", []) if r.get("latest.programs.cip_4_digit")]
        return pd.json_normalize(
            results,
            record_path=["latest.programs.cip_4_digit"],
            meta=META,
            errors="ignore",
        )

    dfs.append(page_to_df(data))

    for page in range(1, total_pages):
        params["page"] = str(page)
        response = get_with_retries(BASE_URL, params=params, timeout=30)
        data = get_json_or_raise(response)
        dfs.append(page_to_df(data))

    df = pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()

    print(f"Total pages fetched: {total_pages}")
    print(f"Total Rows/Programs ingested: {len(df)}")
    return df

def get_json_or_raise(response: requests.Response):
    # Raise for HTTP errors early (4xx/5xx)
    try:
        response.raise_for_status()
    except requests.HTTPError as e:
        ct = response.headers.get("Content-Type", "")
        body_preview = (response.text or "")[:800]
        raise RuntimeError(
            f"HTTP {response.status_code} for {response.url}\n"
            f"Content-Type: {ct}\n"
            f"Body preview:\n{body_preview}"
        ) from e

    # Check content-type sanity (helps catch HTML responses)
    ct = response.headers.get("Content-Type", "")
    if "json" not in ct.lower():
        body_preview = (response.text or "")[:800]
        raise RuntimeError(
            f"Expected JSON but got Content-Type: {ct}\n"
            f"URL: {response.url}\n"
            f"Body preview:\n{body_preview}"
        )

    # Parse JSON with a clearer error if it fails
    try:
        return response.json()
    except json.JSONDecodeError as e:
        body_preview = (response.text or "")[:800]
        raise RuntimeError(
            f"JSON decode failed for {response.url}\n"
            f"Body preview:\n{body_preview}"
        ) from e

def get_with_retries(url, params, tries=5, timeout=30):
    last = None
    for i in range(tries):
        r = requests.get(url, params=params, timeout=timeout, headers={"Accept": "application/json"})
        if r.status_code < 500:
            return r
        last = r
        time.sleep((2 ** i) + random.random())
    return last

# Removed features
---
### 1

```
"latest.student.demographics.avg_family_income"
"latest.student.demographics.median_hh_income"
```
overlaps with ```"latest.student.demographics.median_family_income"```

---
### 2

```
"latest.academics.program_reporter.programs_offered"
```
~83% is null

---
### 3

```
latest.admissions.test_requirements
```
~46% is null and overlaps with ```latest.admissions.admission_rate.overall  ```

---
### 4

```
"latest.admissions.sat_scores.50th_percentile.critical_reading",
"latest.admissions.sat_scores.50th_percentile.math",
"latest.admissions.act_scores.50th_percentile.cumulative",
"latest.admissions.act_scores.50th_percentile.english",
"latest.admissions.act_scores.50th_percentile.math",
"latest.admissions.sat_scores.average.overall",
"latest.admissions.act_scores.midpoint.cumulative"
```
~58% is null and overlaps with ```latest.admissions.admission_rate.overall``` and ```latest.school.open_admissions_policy```



In [ ]:
test=collect(state="")

In [ ]:
tdf = collect()
display(tdf.head())
display(tdf.info())

Total pages fetched: 3
Total Rows/Programs ingested: 2501


,code,title,unit_id,distance,school.type,credential.level,earnings.1_yr.overall_median_earnings,earnings.1_yr.working_not_enrolled.overall_count,earnings.4_yr.overall_median_earnings,earnings.4_yr.working_not_enrolled.overall_count,earnings.5_yr.overall_median_earnings,earnings.5_yr.working_not_enrolled.overall_count,id,school.name,location.lat,location.lon,latest.school.locale,latest.school.carnegie_size_setting,latest.admissions.admission_rate.overall,latest.student.demographics.median_family_income,latest.student.students_with_pell_grant,latest.school.open_admissions_policy,latest.student.demographics.age_entry,latest.school.title_iv.eligibility_type
0,1205,Culinary Arts and Related Services.,132374,1,Public,1,25586.0,22.0,22265,28,33286.0,21.0,132374,Atlantic Technical College,26.24278,-80.192271,21,-2,None,16748,None,1,26,1
1,4603,Electrical and Power Transmission Installers.,132374,2,Public,1,29493.0,32.0,41177,27,NaN,NaN,132374,Atlantic Technical College,26.24278,-80.192271,21,-2,None,16748,None,1,26,1
2,4702,"Heating, Air Conditioning, Ventilation and Ref...",132374,1,Public,1,36966.0,63.0,47325,48,NaN,NaN,132374,Atlantic Technical College,26.24278,-80.192271,21,-2,None,16748,None,1,26,1
3,4706,Vehicle Maintenance and Repair Technologies/Te...,132374,1,Public,1,34269.0,24.0,42839,28,NaN,NaN,132374,Atlantic Technical College,26.24278,-80.192271,21,-2,None,16748,None,1,26,1
4,5108,Allied Health and Medical Assisting Services.,132374,1,Public,1,32976.0,33.0,33081,29,NaN,NaN,132374,Atlantic Technical College,26.24278,-80.192271,21,-2,None,16748,None,1,26,1


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2501 entries, 0 to 2500
Data columns (total 24 columns):
 #   Column                                            Non-Null Count  Dtype  
---  ------                                            --------------  -----  
 0   code                                              2501 non-null   object 
 1   title                                             2501 non-null   object 
 2   unit_id                                           2501 non-null   int64  
 3   distance                                          2501 non-null   int64  
 4   school.type                                       2501 non-null   object 
 5   credential.level                                  2501 non-null   int64  
 6   earnings.1_yr.overall_median_earnings             2100 non-null   float64
 7   earnings.1_yr.working_not_enrolled.overall_count  2100 non-null   float64
 8   earnings.4_yr.overall_median_earnings             2501 non-null   int64  
 9   earnings.4_yr.worki

None

# Why so many missing Admission Rates?

In [6]:
df = tdf.copy()
round(df.isnull()["latest.admissions.admission_rate.overall"].mean(),4)

0.4226

In [7]:
round(df[df["latest.school.open_admissions_policy"]==2].isnull()["latest.admissions.admission_rate.overall"].mean(),4)

0.023

Approximately 40% of institutions have missing values for admission_rate.overall.
According to IPEDS reporting rules, institutions with an open admissions policy do not report traditional selectivity metrics such as admission rate or standardized test scores.

A check confirms that nearly all non-open-admission institutions report admission rates, indicating that the missingness is structural rather than random.

Therefore, missing admission rates are interpreted as corresponding primarily to open-admission institutions, and the open_admissions_policy variable is retained to preserve this structural distinction.

In [8]:
df = df.drop(columns="id")
df = clean(df)

Numeric columns: Index(['distance', 'credential_level', '1_yr_median_earnings',
       '1_yr_working_count', '4_yr_median_earnings', '4_yr_working_count',
       '5_yr_median_earnings', '5_yr_working_count', 'admission_rate_overall',
       'median_family_income', 'students_with_pell_grant'],
      dtype='object')


C:\Users\sebas\PycharmProjects\Git\Seb_branch\institutional-roi-analysis\src\ira\clean\clean_scorecard.py:8: FutureWarning: The default value of regex will change from True to False in a future version. In addition, single character regular expressions will *not* be treated as literal strings when regex=True.
  df.columns = df.columns.str.replace(r".", "_")


In [9]:
df["selectivity_bucket"] = pd.cut(
    df["admission_rate_overall"],
    bins=[0, 0.3, 0.7, 1],
    labels=["elite", "mid", "open"]
)

In [10]:
display(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2501 entries, 0 to 2500
Data columns (total 24 columns):
 #   Column                     Non-Null Count  Dtype   
---  ------                     --------------  -----   
 0   code                       2501 non-null   string  
 1   title                      2501 non-null   string  
 2   unit_id                    2501 non-null   string  
 3   distance                   2501 non-null   int64   
 4   school_type                2501 non-null   string  
 5   credential_level           2501 non-null   int64   
 6   1_yr_median_earnings       2100 non-null   float64 
 7   1_yr_working_count         2100 non-null   float64 
 8   4_yr_median_earnings       2501 non-null   int64   
 9   4_yr_working_count         2501 non-null   int64   
 10  5_yr_median_earnings       1788 non-null   float64 
 11  5_yr_working_count         1788 non-null   float64 
 12  school_name                2501 non-null   string  
 13  location_lat               2501 n

None

In [11]:
df.head()

,code,title,unit_id,distance,school_type,credential_level,1_yr_median_earnings,1_yr_working_count,4_yr_median_earnings,4_yr_working_count,5_yr_median_earnings,5_yr_working_count,school_name,location_lat,location_lon,locale,carnegie_size_setting,admission_rate_overall,median_family_income,students_with_pell_grant,open_admissions_policy,age_entry,title_iv_eligibility_type,selectivity_bucket
0,1205,Culinary Arts and Related Services.,132374,1,Public,1,25586.0,22.0,22265,28,33286.0,21.0,Atlantic Technical College,26.24278,-80.192271,21,-2,NaN,16748.0,NaN,1,26,1,NaN
1,4603,Electrical and Power Transmission Installers.,132374,2,Public,1,29493.0,32.0,41177,27,NaN,NaN,Atlantic Technical College,26.24278,-80.192271,21,-2,NaN,16748.0,NaN,1,26,1,NaN
2,4702,"Heating, Air Conditioning, Ventilation and Ref...",132374,1,Public,1,36966.0,63.0,47325,48,NaN,NaN,Atlantic Technical College,26.24278,-80.192271,21,-2,NaN,16748.0,NaN,1,26,1,NaN
3,4706,Vehicle Maintenance and Repair Technologies/Te...,132374,1,Public,1,34269.0,24.0,42839,28,NaN,NaN,Atlantic Technical College,26.24278,-80.192271,21,-2,NaN,16748.0,NaN,1,26,1,NaN
4,5108,Allied Health and Medical Assisting Services.,132374,1,Public,1,32976.0,33.0,33081,29,NaN,NaN,Atlantic Technical College,26.24278,-80.192271,21,-2,NaN,16748.0,NaN,1,26,1,NaN


In [12]:
save(df,file_type="scorecard",clean=1,file_name="scorecard_FL_programs")
save(tdf,file_name="scorecard_FL_programs")